# 11.g - Heat indicators: threshold mapped (11.b / 11.c) against quantile delta mapping of the daily values

**Ad hoc test.** It reads the inputs of 11.a (BIO calendar files of 1.1.d, crop masks, daily ERA5 and raw EC-Earth3)
and writes only in `test_risk_indicators/<route>/heat_qdm/`. Nothing of 11.a-11.f is changed.

**Question.** 11.b and 11.c apply the threshold of the literature to raw EC-Earth3 after mapping it cell by cell: the
ERA5 quantile q of the threshold in the window is replaced by the EC-Earth3 historical value at q. Counting the days
above the mapped threshold is the same as counting the days above the threshold after a **quantile mapping (QM)** of
the daily values (x -> Q_ERA5(F_hist(x))). QM places a value of a run in the distribution of the **historical** run:
where the tail of EC-Earth3 is narrower or wider than that of ERA5, QM stretches or squeezes the warming of the run
in the tail, so the change of the exceedances can be distorted. **Quantile delta mapping (QDM**, Cannon et al. 2015,
*J. Climate* 28, 6938) keeps the change of the model at each quantile:

x̂ = Q_ERA5(τ) + x - Q_hist(τ), with τ = F_run(x) the quantile of the day in the **run**.

For the historical run both give the same values (τ = F_hist(x)); they differ only in the runs. A simple shift of
the distribution (constant bias) gives the same exceedances with both; a difference of shape does not.

**Method.** Per cell, crop and indicator, in the window of the indicator (as the mapping of 11.b: the days of the
window pooled over all seasons), the distributions of ERA5 1981-2014, EC-Earth3 historical 1981-2014 and of each run
(histograms of 0.1 degC, as 11.a). QDM is applied to the daily values of the window (the correction per 0.1 degC bin and
cell, interpolated between the bin centres), then the threshold of the literature is applied as such.
Indicators: the heat of 11.b (days with Tmax above the critical temperature in phase 2) and the physiological
indicators of 11.c (sterility around anthesis, degree-days above 30 degC with tasmax and tasmin each corrected,
warm nights). Events: indicator above the 75th percentile of the reference seasons, the reference being the
historical run of the same method (identical for QM and QDM up to the binning). Drought (SPEI) is not concerned
(computed on the raw runs, relative to their own reference) and not repeated here.

**Reading.** For every crop, indicator and AR6 zone of 11.a: frequency of the events and the hosing effect
(hosing - no hosing) with QM and with QDM, and their difference. The effects reported in the note (11.b / 11.c) hold
where the difference is small compared with the effect itself.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters (as 11.a)
CHOSEN_VARIANT = "A_60d"          # calendar of the ERA5 BIO files (anthesis = start of phase 2)
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")    # BIO files of 1.1.d (calendar)
REF, REF_PERIOD = "gcm_historical", ("1981", "2014")  # EC-Earth3 historical, raw
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
RUN_PERIOD = {}                   # run -> (first, last year) to read only part of a file; whole file by default
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
HEAT_TCRIT = {"Maize": 35.0, "Wheat": 34.0, "Rice": 36.0, "Soybean": 39.0, "Potato": 30.0}   # degC, 11.a / 11.b
PHYSIO = {                        # 11.c: (name, variable, window, kind, threshold degC)
    "Maize": [("sterility (Tmax > 36 degC, anthesis)", "tasmax", "flower", "days", 36.0),
              ("yield heat (degree-days > 30 degC, season)", "tasmax", "season", "kdd", 30.0)],
    "Soybean": [("yield heat (degree-days > 30 degC, season)", "tasmax", "season", "kdd", 30.0)],
    "Rice": [("sterility (Tmax > 38 degC, anthesis)", "tasmax", "flower", "days", 38.0),
             ("warm nights (mean Tmin, grain filling)", "tasmin", "p2", "mean", None)],
    "Wheat": [("sterility (Tmax > 31 degC, anthesis)", "tasmax", "flower", "days", 31.0),
              ("senescence (Tmax > 34 degC, grain filling)", "tasmax", "p2", "days", 34.0)],
    "Potato": [("warm nights (Tmin > 20 degC, tuber bulking)", "tasmin", "tuber", "days", 20.0)],
}
FLOWER_WINDOW = (-15, 15)         # days around anthesis (start of phase 2): 30 days, Gourdji et al. 2013
TUBER_START = 50                  # potato: tuber initiation, days after sowing (variant C_crop of 1.1.d)
HEAT_PCT = 75                     # event: indicator above this percentile of the reference seasons
N_REGIONS = 6                     # AR6 land regions with the largest area of the crop
TAIL_Q = 0.01                     # QM: beyond the 1st / 99th percentile of ERA5, shift by the difference of that percentile (11.b)
BIN = (-60.0, 70.0, 0.1)          # degC: bins of the histograms (11.a)
METHODS = ("QM", "QDM")

ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
OUT = cu.out_dir(CFG, "test_risk_indicators", ROUTE, "heat_qdm")          # the only folder written
SCEN = ["historical", *RUNS]
SPEC = {"historical": (REF, REF_PERIOD), **{r: (r, RUN_PERIOD.get(r)) for r in RUNS}}

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
base_of = lambda crop: crop.split("_")[0].split("-")[0]
# indicators of each crop: the heat of 11.b, then the physiological ones of 11.c
HEAT = {c: [("heat (Tmax > Tcrit, phase 2)", "tasmax", "p2", "days", HEAT_TCRIT[base_of(c)])] + PHYSIO[base_of(c)]
        for c in CROPS}
# (variable, window) pairs whose daily values are corrected, per crop (degree-days: tasmax and tasmin)
PAIRS_VW = {c: sorted({(v, w) for _, v, w, k, _ in HEAT[c]} | {("tasmin", w) for _, v, w, k, _ in HEAT[c] if k == "kdd"})
            for c in CROPS}

# today's cropland of each crop (as 11.a, RISK_DOMAIN = "cropland"): harvested area, calendar
CROP = {}
for crop in CROPS:
    cm = cu.crop_mask(CFG, crop)
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        cal = {v: ds[v].values for v in ("sowing_doy", "phase2_start_doy", "maturity_doy")}
    known = LAND & np.isfinite(cal["sowing_doy"]) & np.isfinite(cal["maturity_doy"])
    pres = known & (cm["presence"].values == 1)
    if CELL_AREA_THR:
        pres &= cu.cell_area_frac(CFG, cm["area_ha"].values) > CELL_AREA_THR
    CROP[crop] = {"mask": pres, "area": np.nan_to_num(cm["area_ha"].values), "cal": cal}
UNION = np.logical_or.reduce([CROP[c]["mask"] for c in CROPS])
UI, UJ = np.nonzero(UNION)
COL = -np.ones(LAND.shape, int); COL[UI, UJ] = np.arange(UI.size)   # column of each cell in the daily arrays
for crop in CROPS:
    i, j = np.nonzero(CROP[crop]["mask"])
    CROP[crop].update({"ij": (i, j), "col": COL[i, j], "w": CROP[crop]["area"][i, j],
                       **{k: CROP[crop]["cal"][k][i, j].astype(float) for k in CROP[crop]["cal"]}})

try:                                         # AR6 land regions (Iturbide et al. 2020) of the cells
    import regionmask
    AR6 = regionmask.defined_regions.ar6.land
    REG = AR6.mask(GRID["lon"], GRID["lat"]).values
    REG_NAME = dict(zip(AR6.numbers, AR6.abbrevs))
except Exception as e:                       # no regions: global results only
    print("regionmask not available:", e)
    REG, REG_NAME = np.full(LAND.shape, np.nan), {}
print(f"{len(CROPS)} crops, {UI.size} cells | reference {REF} {REF_PERIOD} | runs {RUNS} | written only in {OUT}")
for c in CROPS:
    print(f"  {c:16s} " + " | ".join(h[0] for h in HEAT[c]) + f" | corrected: {PAIRS_VW[c]}")

## 1. Windows, histograms, threshold mapping (functions of 11.a) and QDM from the histograms

In [ ]:
def degree_days_above(tx, tn, b):
    # degree-days above b of one day, single-sine curve between Tmin and Tmax (Baskerville & Emin 1969) (11.a)
    m, a = (tx + tn) / 2, np.maximum((tx - tn) / 2, 1e-6)
    th = np.arcsin(np.clip((b - m) / a, -1, 1))
    return np.maximum(((m - b) * (np.pi / 2 - th) + a * np.cos(th)) / np.pi, 0.0)


def ylen(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365


def windows(dates, c):
    # the seasons of the crop (harvest year hy, with hy - 1 in the data): day indices and masks (days, cells) of the
    # season, phase 2, the 30 days around anthesis, tuber bulking (11.a)
    yr, doy = dates.year.values, dates.dayofyear.values
    years = sorted(set(yr))
    sow, ant, mat = c["sowing_doy"], c["phase2_start_doy"], c["maturity_doy"]
    wrap = mat > 365
    for hy in years:
        if hy - 1 not in years:
            continue
        idx = np.where((yr == hy) | (yr == hy - 1))[0]
        T = (doy[idx] + np.where(yr[idx] == hy, ylen(hy - 1), 0))[:, None]
        shift = np.where(wrap, 0, ylen(hy - 1))[None]
        s, a, m = sow[None] + shift, ant[None] + shift, mat[None] + shift
        yield hy, idx, {"season": (T >= s) & (T <= m), "p2": (T >= a) & (T <= m),
                        "flower": (T >= a + FLOWER_WINDOW[0]) & (T < a + FLOWER_WINDOW[1]),
                        "tuber": (T >= s + TUBER_START) & (T <= m)}


NB = int(round((BIN[1] - BIN[0]) / BIN[2]))
CENTRE = BIN[0] + (np.arange(NB) + 0.5) * BIN[2]


def bin_of(x):
    return np.clip(((x - BIN[0]) / BIN[2]).astype(int), 0, NB - 1)


def counts(X, dates, c, wnd):
    # histogram (cells, bins) of the daily values of X in the window, over all the seasons (11.a)
    n, col = c["col"].size, c["col"]
    cnt = np.zeros(n * NB, np.int64)
    for _, idx, W in windows(dates, c):
        b = bin_of(X[idx][:, col])
        cnt += np.bincount((np.arange(n)[None] * NB + b)[W[wnd]], minlength=n * NB)
    return cnt.reshape(n, NB)


def cdf(cnt):
    return np.cumsum(cnt, 1) / np.maximum(cnt.sum(1, keepdims=True), 1)


def value_at(cnt, q):
    # value at the quantile q (per cell): upper edge of the first bin where the empirical CDF reaches q (11.a)
    return BIN[0] + (np.argmax(cdf(cnt) >= np.asarray(q)[:, None] - 1e-12, axis=1) + 1) * BIN[2]


def mapped_threshold(cnt_e, cnt_m, t0):
    # 11.b: threshold of EC-Earth3 = its quantile q of the window, q = quantile of t0 in ERA5; beyond the 1st / 99th
    # percentile of ERA5: t0 shifted by the difference of that percentile (EC-Earth3 - ERA5)
    n = cnt_e.shape[0]
    q = cdf(cnt_e)[np.arange(n), int(np.clip((t0 - BIN[0]) / BIN[2], 0, NB - 1))]
    thr = value_at(cnt_m, np.clip(q, TAIL_Q, 1 - TAIL_Q))
    for tail, sel in ((1 - TAIL_Q, q > 1 - TAIL_Q), (TAIL_Q, q < TAIL_Q)):
        qq = np.full(n, tail)
        thr = np.where(sel, t0 + value_at(cnt_m, qq) - value_at(cnt_e, qq), thr)
    return thr, q


def quantile(C, tau):
    # quantile function of one cell from its CDF at the upper bin edges (C, NB), linear within the bin
    b = np.minimum(np.searchsorted(C, tau, side="left"), NB - 1)
    lo = np.where(b > 0, C[np.maximum(b - 1, 0)], 0.0)
    frac = np.where(C[b] > lo, (tau - lo) / np.maximum(C[b] - lo, 1e-12), 1.0)
    return BIN[0] + (b + np.clip(frac, 0, 1)) * BIN[2]


def qdm_table(cnt_e, cnt_h, cnt_r):
    # QDM (Cannon et al. 2015, additive) of the run r against ERA5, calibrated on the historical run h, per cell: the
    # correction Q_e(tau) - Q_h(tau) at the centre of each bin of the run, tau = mid-rank of the bin in the run
    # (cells, NB); bins absent from the run take the correction of the nearest bin of the run (only used between two
    # bin centres, see apply_table)
    Ce, Ch, Cr = cdf(cnt_e), cdf(cnt_h), cdf(cnt_r)
    corr = np.zeros(cnt_r.shape, np.float32)
    for k in range(cnt_r.shape[0]):
        used = np.nonzero(cnt_r[k])[0]
        if used.size == 0 or cnt_e[k].sum() == 0 or cnt_h[k].sum() == 0:
            continue
        tau = (np.where(used > 0, Cr[k, np.maximum(used - 1, 0)], 0.0) + Cr[k, used]) / 2
        corr[k] = np.interp(np.arange(NB), used, quantile(Ce[k], tau) - quantile(Ch[k], tau))
    return corr


def apply_table(corr, x):
    # corrected daily values (days, cells): x + the correction of the cell interpolated linearly between bin centres
    pos = (x - BIN[0]) / BIN[2] - 0.5
    b0 = np.clip(np.floor(pos).astype(int), 0, NB - 2)
    f = np.clip(pos - b0, 0.0, 1.0)
    c = np.arange(x.shape[1])[None]
    return x + corr[c, b0] * (1 - f) + corr[c, b0 + 1] * f

## 2. Indicators per season with both methods

QM: the raw daily values against the mapped threshold of 11.b (degree-days: the mapped threshold applied to the raw
tasmax and tasmin, as 11.c); QDM: the corrected daily values against the threshold of the literature. Warm nights
(mean Tmin, relative event): the mean of the window, raw for QM (as 11.c), corrected for QDM.

In [ ]:
def seasons_qm(arr, dates, c, heat, thr):
    # 11.a seasons() without the water balance: per season and cell, the heat indicators with one threshold per cell
    col = c["col"]
    hys, out = [], {h[0]: [] for h in heat}
    for hy, idx, W in windows(dates, c):
        X = {v: arr[v][idx][:, col] for v in arr}
        for name, var, wnd, kind, _ in heat:
            m = W[wnd]
            if kind == "days":
                out[name].append(((X[var] > thr[name]) & m).sum(0))
            elif kind == "kdd":
                out[name].append((degree_days_above(X["tasmax"], X["tasmin"], thr[name]) * m).sum(0))
            else:
                out[name].append(np.where(m, X[var], 0.0).sum(0) / np.maximum(m.sum(0), 1))
        hys.append(hy)
    return np.array(hys), {k: np.array(v, float) for k, v in out.items()}


def seasons_qdm(arr, dates, c, heat, luts):
    # the same indicators on the daily values corrected by QDM in the window of each indicator, threshold as such
    col = c["col"]
    hys, out = [], {h[0]: [] for h in heat}
    for hy, idx, W in windows(dates, c):
        X = {v: arr[v][idx][:, col] for v in arr}
        for name, var, wnd, kind, t0 in heat:
            m = W[wnd]
            if kind == "days":
                out[name].append(((apply_table(luts[(var, wnd)], X[var]) > t0) & m).sum(0))
            elif kind == "kdd":
                tx, tn = apply_table(luts[("tasmax", wnd)], X["tasmax"]), apply_table(luts[("tasmin", wnd)], X["tasmin"])
                out[name].append((degree_days_above(tx, np.minimum(tn, tx), t0) * m).sum(0))
            else:
                out[name].append(np.where(m, apply_table(luts[(var, wnd)], X[var]), 0.0).sum(0) / np.maximum(m.sum(0), 1))
        hys.append(hy)
    return np.array(hys), {k: np.array(v, float) for k, v in out.items()}


def read_daily(name, period, vars_=("tasmax", "tasmin")):
    t0 = time.time()
    arr = {}
    for v in vars_:
        da = cu.read_climate(CFG, name, v, GRID, period)
        arr[v] = cu.land_series(da, (UI, UJ))
    dates = pd.DatetimeIndex(da["time"].values).normalize()
    print(f"  {name} {period or 'whole file'}: {dates[0].date()} -> {dates[-1].date()} ({(time.time() - t0) / 60:.1f} min)", flush=True)
    return arr, dates


def histograms(arr, dates):
    return {(c, v, w): counts(arr[v], dates, CROP[c], w) for c in CROPS for v, w in PAIRS_VW[c]}


arr, dates = read_daily("era5", ERA5_PERIOD)
CNT_E = histograms(arr, dates)
del arr
SEAS, CNT_H, TH, ROWS = {}, None, {}, []
for s in SCEN:
    name, period = SPEC[s]
    arr, dates = read_daily(name, period)
    cnt = histograms(arr, dates)
    if s == "historical":
        CNT_H = cnt
        for crop in CROPS:                   # thresholds of 11.b / 11.c, mapped on the historical run
            TH[crop] = {}
            for name_, var, wnd, kind, t0 in HEAT[crop]:
                TH[crop][name_] = (np.full(CROP[crop]["col"].size, np.nan) if t0 is None else
                                   mapped_threshold(CNT_E[(crop, var, wnd)], cnt[(crop, var, wnd)], t0)[0])
    for crop in CROPS:
        luts = {(v, w): qdm_table(CNT_E[(crop, v, w)], CNT_H[(crop, v, w)], cnt[(crop, v, w)]) for v, w in PAIRS_VW[crop]}
        SEAS[("QM", s, crop)] = seasons_qm(arr, dates, CROP[crop], HEAT[crop], TH[crop])
        SEAS[("QDM", s, crop)] = seasons_qdm(arr, dates, CROP[crop], HEAT[crop], luts)
        w = CROP[crop]["w"]
        for h in HEAT[crop]:
            ROWS.append({"crop": crop, "indicator": h[0], "scenario": s,
                         **{f"{m}, mean per season": np.average(SEAS[(m, s, crop)][1][h[0]].mean(0), weights=w) for m in METHODS}})
    del arr
    print(f"{s}: {len(SEAS[('QM', s, CROPS[0])][0])} seasons", flush=True)
MEAN = pd.DataFrame(ROWS).set_index(["crop", "indicator", "scenario"])
MEAN["QDM - QM"] = MEAN["QDM, mean per season"] - MEAN["QM, mean per season"]
MEAN.to_csv(OUT / "indicator_means_qm_qdm.csv")
with pd.option_context("display.max_rows", 300, "display.width", 250):
    display(MEAN.round(3))
# check: on the historical run QDM is QM (up to the binning and the tail rule of the mapping)
H = MEAN.xs("historical", level="scenario")
rel = (H["QDM - QM"].abs() / H["QM, mean per season"].abs().clip(lower=1e-6))
print("historical, |QDM - QM| / QM per indicator: max %.3f, median %.3f" % (rel.max(), rel.median()))

## 3. Events relative to the reference (P75 of the reference seasons of each method), per zone

In [ ]:
def zones_of(crop):
    # (name, cells, weights): today's cropland globally and in its N_REGIONS largest AR6 regions (harvested area) (11.a)
    c = CROP[crop]
    i, j = c["ij"]
    reg = REG[i, j]
    tot = pd.Series(c["w"]).groupby(reg).sum().sort_values(ascending=False) if REG_NAME else pd.Series(dtype=float)
    top = [int(r) for r in tot.index[:N_REGIONS] if np.isfinite(r)]
    return [("global", np.ones(i.size, bool), c["w"])] + [(REG_NAME[r], reg == r, c["w"]) for r in top]


EV = {}
for m in METHODS:
    for crop in CROPS:
        _, ref = SEAS[(m, "historical", crop)]
        p_ref = {h[0]: np.percentile(ref[h[0]], HEAT_PCT, axis=0) for h in HEAT[crop]}
        for s in SCEN:
            _, x = SEAS[(m, s, crop)]
            EV[(m, s, crop)] = {n: x[n] > p[None] for n, p in p_ref.items()}

rows = []
for crop in CROPS:
    for zname, z, w in zones_of(crop):
        for h in HEAT[crop]:
            for m in METHODS:
                f = {s: 100 * np.average(EV[(m, s, crop)][h[0]].mean(0)[z], weights=w[z]) for s in SCEN}
                rows.append({"crop": crop, "zone": zname, "indicator": h[0], "method": m, **f,
                             **{f"{a} - {b}": f[a] - f[b] for a, b in PAIRS},
                             **{f"{r} - historical": f[r] - f["historical"] for r in RUNS}})
FREQ = pd.DataFrame(rows).set_index(["crop", "zone", "indicator", "method"])
FREQ.to_csv(OUT / "event_frequency_qm_qdm.csv")
EFF = [f"{a} - {b}" for a, b in PAIRS]
CMP = FREQ[EFF + [f"{r} - historical" for r in RUNS]].unstack("method")
for col in EFF + [f"{r} - historical" for r in RUNS]:
    CMP[(col, "QDM - QM")] = CMP[(col, "QDM")] - CMP[(col, "QM")]
CMP = CMP.sort_index(axis=1)
CMP.to_csv(OUT / "hosing_effect_qm_vs_qdm.csv")
with pd.option_context("display.max_rows", 500, "display.width", 250, "display.max_columns", 40):
    display(CMP[[c for c in CMP.columns if c[0] in EFF]].round(1))

## 4. Summary: does the hosing effect depend on the method?

Per crop and indicator, over its zones: the largest |hosing effect| (QM) and the largest |QDM - QM| of the effect
(points of % of the seasons), and the zones where the method changes the sign of an effect of at least 5 points or
moves it by more than 5 points.

In [ ]:
rows = []
for (crop, zone, ind), r in CMP.iterrows():
    for e in EFF:
        qm, qdm = r[(e, "QM")], r[(e, "QDM")]
        rows.append({"crop": crop, "zone": zone, "indicator": ind, "pair": e, "QM": qm, "QDM": qdm, "QDM - QM": qdm - qm,
                     "sign changes": bool(max(abs(qm), abs(qdm)) >= 5 and np.sign(qm) != np.sign(qdm)),
                     "moved > 5 points": bool(abs(qdm - qm) > 5)})
DIFF = pd.DataFrame(rows)
DIFF.to_csv(OUT / "hosing_effect_differences.csv", index=False)
SUM = DIFF.groupby(["crop", "indicator"]).agg(max_abs_effect_QM=("QM", lambda x: x.abs().max()),
                                               max_abs_QDM_minus_QM=("QDM - QM", lambda x: x.abs().max()),
                                               sign_changes=("sign changes", "sum"), moved_over_5=("moved > 5 points", "sum"),
                                               cases=("QM", "size"))
display(SUM.round(1))
flag = DIFF[DIFF["sign changes"] | DIFF["moved > 5 points"]]
print(f"{len(flag)} of {len(DIFF)} (zone, indicator, pair) cases where the method matters (sign or > 5 points):")
with pd.option_context("display.max_rows", 200, "display.width", 250):
    display(flag.round(1))

## 5. Maps (maize, 2050): hosing effect with QM, with QDM, and the difference

In [ ]:
crop = "Maize_rf" if "Maize_rf" in CROPS else CROPS[0]
i, j = CROP[crop]["ij"]
h, o = PAIRS[-1]


def grid_of(x):
    g = np.full(LAND.shape, np.nan)
    g[i, j] = x
    return g


fig, axes = plt.subplots(len(HEAT[crop]), 3, figsize=(30, 5.2 * len(HEAT[crop])), squeeze=False)
for r, ind in enumerate([x[0] for x in HEAT[crop]]):
    eff = {m: 100 * (EV[(m, h, crop)][ind].mean(0) - EV[(m, o, crop)][ind].mean(0)) for m in METHODS}
    for k, (val, title, lim) in enumerate([(eff["QM"], "QM (threshold mapped)", 30), (eff["QDM"], "QDM", 30),
                                           (eff["QDM"] - eff["QM"], "QDM - QM", 15)]):
        ax = axes[r, k]
        ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, 0, np.nan), cmap="Greys", vmin=0, vmax=4, shading="nearest")
        im = ax.pcolormesh(GRID["lon"], GRID["lat"], grid_of(val), cmap="RdBu_r", vmin=-lim, vmax=lim, shading="nearest")
        ax.set_ylim(-60, 75); ax.set_title(f"{crop}, {ind}: {h} - {o}, {title} (points of % of the seasons)", fontsize=10)
        plt.colorbar(im, ax=ax, shrink=0.8)
fig.savefig(OUT / f"hosing_effect_maps_qm_qdm_{crop}.png", dpi=90, bbox_inches="tight"); plt.show()